# Data pipeline — from COMSOL exports to sorted modes

This notebook is the **entry point** to the repository. It explains how the 32
files in `data/` are structured and how the eigenvalues are turned into the
mode-sorted tables used by the figure notebooks.

**What you will do**

1. Read one COMSOL export and inspect its header.
2. Load all 32 files and build a provenance table.
3. Sort the eigenvalues into modes (`VP_1`, `VP_2`, ...).
4. Plot a first quick spectrum to check the pipeline.
5. Learn the units and the two pitfalls (unordered eigenvalues, mixed units).

> **No simulation is run here.** Everything is post-processing of existing
> COMSOL exports.


## Step 0 — imports and paths

The helper module `chain_utils.py` lives next to this notebook and contains the
three functions used below: `read_comsol`, `load_all` and `organize`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import chain_utils as cu

DATA = cu.DATA
print("data folder:", DATA.resolve())
print("exports found:", len(list(DATA.glob("*.txt"))))


## Step 1 — anatomy of one COMSOL export

Every `.txt` has exactly **8 header lines** and two data columns:

```
% Model:              caso 1.mph
% Version:            COMSOL 6.3.0.290
% Date:               Jul 11 2026, 15:11
% Dimension:          1
% Nodes:              4515
% Expressions:        1
% Description:        ...
% Length unit:        m
<X> <eigenvalue>
```

The header is the **provenance record**: model name, COMSOL version and export
date. The second column is the eigenvalue in **eV**.


In [ ]:
df, header = cu.read_comsol(DATA / "caso1.txt")
print(header)
df.head()


## Step 2 — load everything and check provenance

The returned `provenance` table is the same information stored in
`data/manifest.csv`. It is useful to verify that no file is missing or
duplicated before plotting.


In [ ]:
files, provenance = cu.load_all()
print("files:", len(files), "| total rows:", sum(len(d) for d in files.values()))
provenance.groupby(["model", "version"]).size().sort_values(ascending=False).head(10)


In [ ]:
provenance.head(8)


## Step 3 — why sorting the eigenvalues is mandatory

COMSOL does **not** return the eigenvalues ordered at each sweep point. If you
plot them as they come, the curves cross and the mode identity is lost.

`organize` sorts by `(X, eigenvalue)`, assigns `mode = cumcount()` inside each
`X`, and pivots the table. The result has one row per sweep point and one column
per mode:

```
X   VP_1   VP_2   VP_3 ...
```


In [ ]:
org = cu.organize(files["caso1.txt"])
print(org.shape)
org.head()


## Step 4 — quick sanity plot

Ground state (`VP_1`) and first excited state (`VP_2`) for the uniform chain
with `d = 0 nm`. The x-axis is converted from metres to nanometres.

**Look for:** the ground state is almost flat at **-21.4 meV** and the first
excited state oscillates with the opposite phase. That is the s/p symmetry rule
explained in `docs/metodo.md`.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x_nm = org["X"] * 1e9

for col in ["VP_1", "VP_2", "VP_3", "VP_4"]:
    ax.plot(x_nm, org[col], lw=2.2, label=col)

ax.set_xlabel(r"$x_i$ (nm)")
ax.set_ylabel("Energy (eV)")
ax.set_title("Uniform chain, $d = 0$ nm: first four modes")
ax.legend(frameon=False, ncol=4)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Step 5 — units and pitfalls

| Dataset family | `X` meaning | Unit |
|---|---|---|
| `caso*.txt`, `EvsxiR5=*.txt` | impurity position `x_i` | m |
| `EvsR5*.txt` | inter-dot distance `R_5` | m |
| `c*_F.txt`, `AEvsF*.txt` | electric field `F` | kV/cm |

**Pitfalls**

1. **Unordered eigenvalues** — always call `organize` before plotting.
2. **Mixed units** — positions are in metres in the raw files; the notebooks
   multiply by `1e9` to plot in nm. Fields are already in kV/cm.
3. **Degenerate or nearly degenerate levels** — if two eigenvalues are very
   close, the mode index is still well defined after sorting, but the
   eigenvectors may be arbitrary linear combinations. The energy figures are
   unaffected; the orbital images are not.
4. **Do not merge the raw files** — the per-file header is the provenance.
   Use `data/manifest.csv` or `scripts/consolidar.py` if you need one table.


## Where to go next

- `02_results_figures.ipynb` — reproduces the final thesis figures.
- `03_dipole_analysis.ipynb` — extracts `p_1`, `p_2` and the critical field.
- `docs/metodo.md` — model, mesh, validation and the s/p symmetry rule.
